In [2]:
import sys
from pathlib import Path

# notebook/jupyter  →  proyecto/
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import scr.python.procesamiento_microdatos as mic

%load_ext autoreload
%autoreload 2

In [ ]:
# Pasar de csvs mensuales a estructura data lake

years_def = range(1995, 2024)   
years_prov = range(2024, 2026) 
raw_base_dir = "../../data/raw"
out_dir = "../../data/rawparquet"

# Configurar filtros de provincia (Madrid)
filtros_provincia = {"madrid": [28]}

print("=== Iniciando procesamiento de data lake ===")
print(f"  - Datos completos: {out_dir}/{{taric|sectores}}/")
print(f"  - Madrid (prov 28): {out_dir}/madrid/{{taric|sectores}}/")
print(f"  - España (agregado): {out_dir}/espana/{{taric|sectores}}/")
print(f"  - Totales CCAA: {out_dir}/totalesccaa/")

# Años definitivos
for year in years_def:
    print(f"\n  Año {year} (def)")
    for month in range(1, 13):
        try:
            mic.csvs_a_data_lake(
                year=year,
                month=month,
                version="def",
                raw_base_dir=raw_base_dir,
                out_dir=out_dir,
                filtros_provincia=filtros_provincia
            )
        except FileNotFoundError:
            print(f"    [SKIP] No existe CSV para {year}-{month:02d}")

# Años provisionales
for year in years_prov:
    print(f"\n  Año {year} (prov)")
    for month in range(1, 13):
        try:
            mic.csvs_a_data_lake(
                year=year,
                month=month,
                version="prov",
                raw_base_dir=raw_base_dir,
                out_dir=out_dir,
                filtros_provincia=filtros_provincia
            )
        except FileNotFoundError:
            print(f"    [SKIP] No existe CSV para {year}-{month:02d}")

print(f"\n[COMPLETADO] Data lake creado en {out_dir}")
print("\nEstructura generada:")
print(f"  • {out_dir}/taric/           → Datos completos TARIC")
print(f"  • {out_dir}/sectores/        → Datos completos Sectores")
print(f"  • {out_dir}/madrid/taric/    → Madrid TARIC (prov 28)")
print(f"  • {out_dir}/madrid/sectores/ → Madrid Sectores (prov 28)")
print(f"  • {out_dir}/espana/taric/    → España agregado TARIC")
print(f"  • {out_dir}/espana/sectores/ → España agregado Sectores")
print(f"  • {out_dir}/totalesccaa/     → Totales por CCAA")

In [3]:
# Añadir un mes nuevo al data lake
year = 2025
month = 10
raw_base_dir = "../../data/raw"
out_dir = "../../data/rawparquet"

# Configurar filtros de provincia (Madrid)
filtros_provincia = {"madrid": [28]}

print(f"=== Añadiendo mes nuevo: {year}-{month:02d} (provisional) ===")
print(f"  - Datos completos: {out_dir}/{{taric|sectores}}/")
print(f"  - Madrid (prov 28): {out_dir}/madrid/{{taric|sectores}}/")
print(f"  - España (agregado): {out_dir}/espana/{{taric|sectores}}/")
print(f"  - Totales CCAA: {out_dir}/totalesccaa/\n")

try:
    mic.csvs_a_data_lake(
        year=year,
        month=month,
        version="prov",
        raw_base_dir=raw_base_dir,
        out_dir=out_dir,
        filtros_provincia=filtros_provincia
    )
    print(f"\n✓ Procesamiento completado exitosamente")
    print(f"✓ Se han generado 7 datasets:")
    print(f"  • taric (completo)")
    print(f"  • sectores (completo)")
    print(f"  • madrid/taric")
    print(f"  • madrid/sectores")
    print(f"  • espana/taric")
    print(f"  • espana/sectores")
    print(f"  • totalesccaa")
    
except FileNotFoundError as e:
    print(f"\n✗ CSV no encontrado: {e}")
    
except Exception as e:
    print(f"\n✗ Error durante el procesamiento: {e}")

print(f"\n=== Completado ===")

=== Añadiendo mes nuevo: 2025-10 (provisional) ===
  - Datos completos: ../../data/rawparquet/{taric|sectores}/
  - Madrid (prov 28): ../../data/rawparquet/madrid/{taric|sectores}/
  - España (agregado): ../../data/rawparquet/espana/{taric|sectores}/
  - Totales CCAA: ../../data/rawparquet/totalesccaa/

    → ../../data/rawparquet/taric/estado=0/anio=2025/mes=10/
    → ../../data/rawparquet/madrid/taric/estado=0/anio=2025/mes=10/ (provincias: [28])
    → ../../data/rawparquet/espana/taric/estado=0/anio=2025/mes=10/
[OK] Procesado comex_taric_202510.csv en todas sus versiones
    → ../../data/rawparquet/sectores/estado=0/anio=2025/mes=10/
    → ../../data/rawparquet/madrid/sectores/estado=0/anio=2025/mes=10/ (provincias: [28])
    → ../../data/rawparquet/espana/sectores/estado=0/anio=2025/mes=10/
    → ../../data/rawparquet/totalesccaa/estado=0/anio=2025/mes=10/
[OK] Procesado comex_sec_202510.csv en todas sus versiones

✓ Procesamiento completado exitosamente
✓ Se han generado 7 datase

In [ ]:
# Actualizar año completo de provisional a definitivo
year = 2023 
raw_base_dir = "../../data/raw"
out_dir = "../../data/rawparquet"

# Configurar filtros de provincia (Madrid)
filtros_provincia = {"madrid": [28]}

print(f"=== Actualizando {year} de provisional a definitivo ===")
print(f"  - Generando: completos, madrid, españa y totalesccaa")
print(f"  - Procesando: taric y sectores automáticamente\n")

errores = []

for month in range(1, 13):
    try:
        print(f"Procesando {year}-{month:02d}...")
        mic.csvs_a_data_lake(
            year=year,
            month=month,
            version="def",
            raw_base_dir=raw_base_dir,
            out_dir=out_dir,
            filtros_provincia=filtros_provincia
        )
        print(f"  ✓ {year}-{month:02d}: OK (7 datasets generados)\n")
        
    except FileNotFoundError as e:
        error_msg = f"{year}-{month:02d}: CSV no encontrado - {e}"
        errores.append(error_msg)
        print(f"  ✗ {year}-{month:02d}: CSV no encontrado\n")
        
    except Exception as e:
        error_msg = f"{year}-{month:02d}: {e}"
        errores.append(error_msg)
        print(f"  ✗ {year}-{month:02d}: Error - {e}\n")

print(f"=== Actualización completada ===")
print(f"\nResumen:")
print(f"  • Meses procesados exitosamente: {12 - len(errores)}/12")
print(f"  • Datasets por mes: 7 (taric, sectores, madrid/taric, madrid/sectores, espana/taric, espana/sectores, totalesccaa)")
print(f"  • Total datasets generados: {(12 - len(errores)) * 7}")

if errores:
    print(f"\n⚠️  Se encontraron {len(errores)} errores:")
    for error in errores:
        print(f"  - {error}")
else:
    print(f"\n✓ Año {year} actualizado completamente sin errores")